In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [2]:
words = open('names.txt', 'r').read().splitlines()

In [3]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}


In [4]:
VOCAB_SIZE   = 27
CONTEXT_SIZE = 3    
EMBED_DIM    = 10   
N_HIDDEN     = 200  

In [5]:
def build_dataset(words):
    X, Y = [], []
    for w in words:
        context = [0] * CONTEXT_SIZE
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    return torch.tensor(X), torch.tensor(Y)

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr, Ytr   = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte   = build_dataset(words[n2:])


In [6]:
#Model parameters
g = torch.Generator().manual_seed(2147483647)

C  = torch.randn((VOCAB_SIZE, EMBED_DIM),              generator=g)  # embedding table
W1 = torch.randn((CONTEXT_SIZE * EMBED_DIM, N_HIDDEN), generator=g) * 0.01
b1 = torch.randn((N_HIDDEN,),                          generator=g) * 0.01
W2 = torch.randn((N_HIDDEN, VOCAB_SIZE),               generator=g) * 0.01
b2 = torch.randn((VOCAB_SIZE,),                        generator=g) * 0.01

# MODIFICATION
W3 = torch.randn((CONTEXT_SIZE * EMBED_DIM, VOCAB_SIZE), generator=g) * 0.01

parameters = [C, W1, b1, W2, b2, W3]
print(f"{sum(p.nelement() for p in parameters)}")

for p in parameters:
  p.requires_grad = True

12707


In [9]:
for i in range(30000):
    # minibatch
    ix = torch.randint(0, Xtr.shape[0], (64,), generator=g)

    # forward pass
    emb  = C[Xtr[ix]]                     # (64, 3, 10)
    x    = emb.view(emb.shape[0], -1)     # (64, 30)  ← flattened embeddings
    h    = torch.tanh(x @ W1 + b1)        # (64, 200) ← non-linear path
    
    # MODIFICATION
    logits = h @ W2 + x @ W3 + b2        # (64, 27)
    #               ↑ this is the direct connection

    loss = F.cross_entropy(logits, Ytr[ix])

    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    # update
    lr = 0.1 if i < 20000 else 0.01
    for p in parameters:
        p.data += -lr * p.grad
    if i % 10000 == 0: # print every once in a while
        print(f'{i:7d}/{30000}: {loss.item():.4f}')
  

      0/30000: 3.2910
  10000/30000: 2.2960
  20000/30000: 2.4621


In [10]:
def evaluate(X, Y):
    emb = C[X]
    x   = emb.view(emb.shape[0], -1)
    h   = torch.tanh(x @ W1 + b1)
    logits = h @ W2 + x @ W3 + b2
    return F.cross_entropy(logits, Y).item()

print(f"Train loss: {evaluate(Xtr, Ytr)}")
print(f"Dev   loss: {evaluate(Xdev, Ydev)}")

Train loss: 2.2707748413085938
Dev   loss: 2.2752153873443604
